## Imports

In [1]:
import glob
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

import time

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='processing-2.processing.data.production.internal' ,
    # 'presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

presto_port = '80'
username = 'manoj.ravirajan@rapido.bike'
conn1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
conn2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
conn3= presto.connect('processing-2.processing.data.production.internal',presto_port,username)
conn4= presto.connect('processing.processing.data.production.internal',presto_port,username)

In [4]:
local_path = '/Users/E2074/local-datasets/customer/segmentation/consideration/'
local_path

'/Users/E2074/local-datasets/customer/segmentation/consideration/'

In [5]:
city = 'delhi'
city_name = 'Delhi'

## Ignore following steps and go to <b>Customer Segment</b> Section directly

## Datasets & Parameter

In [7]:
start_date = datetime.strptime("2025-03-03", "%Y-%m-%d")

run_date = [(start_date + timedelta(weeks=i)).strftime("%Y-%m-%d") for i in range(10)]
run_date

['2025-03-03',
 '2025-03-10',
 '2025-03-17',
 '2025-03-24',
 '2025-03-31',
 '2025-04-07',
 '2025-04-14',
 '2025-04-21',
 '2025-04-28',
 '2025-05-05']

In [23]:
def get_segment_customer_base(run_date):
    start_time = time.time()

    ltr_segment = ['HH', 'PHH']
    df_list = []
    
    for segment in ltr_segment:
        base_query = f"""
    
            with base as (
                select 
                    customer_id,
                    taxi_ltr_segment,
                    -- taxi_lifetime_rides,
                    date_diff( 'day', DATE(taxi_lifetime_first_ride_date), DATE(DATE_PARSE(run_date, '%Y-%m-%d'))) rapido_firstride_age
                from 
                    datasets.iallocator_customer_segments
                where 
                    run_date = date_format(date('{run_date}'), '%Y-%m-%d')
                    and taxi_lifetime_last_ride_city = '{city_name}'
                    and taxi_ltr_segment = '{segment}'
                ),
                
                active_week as (
                
                select
                    customerid,
                    91 max_active_days,
                    total_active_days,
                    isAct_w01,
                    isAct_w02,
                    isAct_w03,
                    isAct_w04,
                    isAct_w05,
                    isAct_w06,
                    isAct_w07,
                    isAct_w08,
                    isAct_w09,
                    isAct_w10,
                    isAct_w11,
                    isAct_w12,
                    isAct_w13,
                    13 max_active_weeks,
                    (isAct_w01 + isAct_w02 + isAct_w03 + isAct_w04 + isAct_w05 + isAct_w06 + isAct_w07 + isAct_w08 + isAct_w09 + isAct_w10 + isAct_w11 + isAct_w12 + isAct_w13) total_active_weeks,
                    act_days_w01,
                    act_days_w02,
                    act_days_w03,
                    act_days_w04,
                    act_days_w05,
                    act_days_w06,
                    act_days_w07,
                    act_days_w08,
                    act_days_w09,
                    act_days_w10,
                    act_days_w11,
                    act_days_w12,
                    act_days_w13,
                    net_rides
                from
                    (
                    select
                        customerid,
                        sum(rr_active_day) total_active_days,
                        max(case when day >= date_format(date('{run_date}') - interval '91' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '85' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w01,
                        max(case when day >= date_format(date('{run_date}') - interval '84' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '78' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w02,
                        max(case when day >= date_format(date('{run_date}') - interval '77' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '71' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w03,
                        max(case when day >= date_format(date('{run_date}') - interval '70' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '64' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w04,
                        max(case when day >= date_format(date('{run_date}') - interval '63' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '57' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w05,
                        max(case when day >= date_format(date('{run_date}') - interval '56' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '50' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w06,
                        max(case when day >= date_format(date('{run_date}') - interval '49' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '43' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w07,
                        max(case when day >= date_format(date('{run_date}') - interval '42' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '36' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w08,
                        max(case when day >= date_format(date('{run_date}') - interval '35' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '29' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w09,
                        max(case when day >= date_format(date('{run_date}') - interval '28' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '22' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w10,
                        max(case when day >= date_format(date('{run_date}') - interval '21' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '15' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w11,
                        max(case when day >= date_format(date('{run_date}') - interval '14' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '8' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w12,
                        max(case when day >= date_format(date('{run_date}') - interval '7' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '1' day, '%Y-%m-%d')  and rr_active_day > 0 then 1 else 0 end ) isAct_w13,
                        
                        count(distinct case when day >= date_format(date('{run_date}') - interval '91' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '85' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w01,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '84' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '78' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w02,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '77' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '71' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w03,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '70' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '64' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w04,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '63' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '57' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w05,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '56' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '50' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w06,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '49' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '43' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w07,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '42' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '36' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w08,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '35' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '29' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w09,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '28' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '22' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w10,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '21' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '15' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w11,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '14' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '8' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w12,
                        count(distinct case when day >= date_format(date('{run_date}') - interval '7' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') - interval '1' day, '%Y-%m-%d')  and rr_active_day > 0 then day end ) act_days_w13,
                        sum(net_rides_daily) net_rides
                    
                    from    
                        (
                        select 
                            day,
                            customerid, 
                            -- max(fe_sessions_unique_daily) fe_session, 
                            -- sum(rr_sessions_unique_daily) rr_session, 
                            max(case when rr_sessions_unique_daily > 0 then 1 else 0 end) rr_active_day,
                            sum(net_rides_daily) net_rides_daily
                        from 
                            datasets.customer_rf_daily_kpi
                        where 
                            day >= date_format(date('{run_date}') - interval '91' day, '%Y-%m-%d')
                            and day <= date_format(date('{run_date}') - interval '1' day, '%Y-%m-%d')
                            and city = '{city_name}'
                            and service_name in ('Auto','Auto Lite','Auto Pool','AutoPremium','CityAuto','Link','Bike Lite','Bike Pink','CabEconomy','Bike Metro', 'CabPremium', 'Auto NCR')
                            -- and customerid = '6288c6694281e53bbe33d133'
                        group by 1,2
                        )
                    group by 1
                    )
                )
                
                select
                    *
                from
                    base
                left join 
                    active_week
                    on base.customer_id = active_week.customerid
            """

        df_segment = pd.read_sql(base_query, conn4)
        df_segment['run_date'] = run_date
        df_list.append(df_segment)
        print(f"Fetched data for segment: {segment}")
    
    # Concatenate all segment dataframes
    df_base_query = pd.concat(df_list, ignore_index=True)

    df_base_query.to_parquet(local_path + city + '/customer_base_{}.parquet'.format(run_date), index=False)
    end_time = time.time()
    print(f"Executed - Time taken: {end_time - start_time:.4f} seconds")

In [25]:
for i in run_date[9:]:
    
    print('Starting - ', i)
    get_segment_customer_base(i)

Starting -  2025-05-05
Fetched data for segment: HH
Fetched data for segment: PHH
Executed - Time taken: 781.3260 seconds


In [26]:
# Find all matching parquet files
file_paths = glob.glob(os.path.join(local_path + city + '/customer_base_*.parquet'))

# Dictionary to store dynamically created DataFrames
dfs = {}

for path in file_paths:
    filename = os.path.basename(path)
    # Get last 10 characters of the filename without extension
    name_part = os.path.splitext(filename)[0][-10:]
    # Remove special characters
    cleaned_name = re.sub(r'\W+', '', name_part)
    # Create a dynamic dataframe name
    df_name = f'df_{cleaned_name}'
    # Read the parquet file
    dfs[df_name] = pd.read_parquet(path)

In [27]:
df_list = list(dfs.keys())
df_list

['df_20250414',
 'df_20250331',
 'df_20250317',
 'df_20250428',
 'df_20250407',
 'df_20250421',
 'df_20250505',
 'df_20250310',
 'df_20250324',
 'df_20250303']

## Feature Engineering

In [28]:
def get_feature_engineering(df):
    
    print('feature engineering part')
    # Adding required active month fields
    df['isAct_m01'] = np.where(df[['isAct_w01', 'isAct_w02', 'isAct_w03', 'isAct_w04']].sum(axis=1) >= 1, 1, 0)
    df['isAct_m02'] = np.where(df[['isAct_w05', 'isAct_w06', 'isAct_w07', 'isAct_w08']].sum(axis=1) >= 1, 1, 0)
    df['isAct_m03'] = np.where(df[['isAct_w09', 'isAct_w10', 'isAct_w11', 'isAct_w12', 'isAct_w13']].sum(axis=1) >= 1, 1, 0)
    
    df['max_active_months'] = 3
    df['total_active_months'] = df[['isAct_m01', 'isAct_m02', 'isAct_m03']].sum(axis=1)
    
    df['days_since_1st_ride_in_3months'] = np.where(df['rapido_firstride_age']>=91,91,df['rapido_firstride_age'])
    df['weeks_since_1st_ride_in_3months'] = np.ceil(df['days_since_1st_ride_in_3months']/7)
    df['months_since_1st_ride_in_3months'] = np.ceil(df['days_since_1st_ride_in_3months']/31)

    df["customerid"] = df["customerid"].fillna('NA')
    df.fillna(0,inplace=True)


    # Adding required derived percentage fields for making thresholds
    df['active_days_percet'] = (df['total_active_days']/df['days_since_1st_ride_in_3months']).round(2)
    df['active_weeks_percet'] = (df['total_active_weeks']/df['weeks_since_1st_ride_in_3months']).round(2)
    df['active_months_percet'] = (df['total_active_months']/df['months_since_1st_ride_in_3months']).round(2)

    return df

## Segmentation

In [29]:
def get_segment(df):

    print('segmentation part')
    
    df['hh_monthly_segment'] = np.where((df['taxi_ltr_segment'].isin(['HH'])) & (df['active_months_percet'] >= 0.66), '1-HH-MONTHLY', None)
    df['hh_quarterly_segment'] = np.where((df['taxi_ltr_segment'].isin(['HH'])) & (df['total_active_days'] > 0), '2-HH-QUARTERLY', None)
    df['hh_inactive_segment'] = np.where((df['taxi_ltr_segment'].isin(['HH'])), '3-HH-INACTIVE',  None)
    
    df['daily_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH']))& (df['active_days_percet'] >= 0.45), '4-DAILY', None)
    df['weekly_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH'])) & (df['active_weeks_percet'] >= 0.70), '5-WEEKLY', None)
    df['monthly_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH'])) & (df['active_months_percet'] >= 0.66), '6-MONTHLY', None)
    df['quarterly_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH'])) & df['total_active_days'] > 0, '7-QUARTERLY', None)
    df['inactive_segment'] = np.where((df['taxi_ltr_segment'].isin(['PHH'])) & (df['total_active_days'] == 0), '8-INACTIVE', None)
    df['new_segment'] = np.where((~df['taxi_ltr_segment'].isin(['HH', 'PHH'])), '9-NEW', None)
    df['segment'] = df['hh_monthly_segment']\
                        .fillna(df['hh_quarterly_segment'])\
                        .fillna(df['hh_inactive_segment'])\
                        .fillna(df['daily_segment'])\
                        .fillna(df['weekly_segment'])\
                        .fillna(df['monthly_segment'])\
                        .fillna(df['quarterly_segment'])\
                        .fillna(df['inactive_segment'])\
                        .fillna(df['new_segment'])\
                        .fillna('check')
    
    df['updated_segment'] = df['segment']
    
    df.loc[((df['updated_segment']=='7-QUARTERLY')& (df['weeks_since_1st_ride_in_3months']<=8)), 'updated_segment'] = '6-MONTHLY'
    df.loc[((df['updated_segment'] == '5-WEEKLY') & (df['weeks_since_1st_ride_in_3months']<=2)), 'updated_segment'] = '6-MONTHLY'

    return df

## Execution

In [30]:
df_list

['df_20250414',
 'df_20250331',
 'df_20250317',
 'df_20250428',
 'df_20250407',
 'df_20250421',
 'df_20250505',
 'df_20250310',
 'df_20250324',
 'df_20250303']

In [31]:
for df_name in df_list:
    print(df_name)
    df = dfs[df_name]
    df_temp = get_feature_engineering(df)
    df_final = get_segment(df_temp)
    df_final.to_parquet(local_path + city + '/segment/{}.parquet'.format(df_name), index=False)
    
    print('---')

df_20250414
feature engineering part
segmentation part
---
df_20250331
feature engineering part
segmentation part
---
df_20250317
feature engineering part
segmentation part
---
df_20250428
feature engineering part
segmentation part
---
df_20250407
feature engineering part
segmentation part
---
df_20250421
feature engineering part
segmentation part
---
df_20250505
feature engineering part
segmentation part
---
df_20250310
feature engineering part
segmentation part
---
df_20250324
feature engineering part
segmentation part
---
df_20250303
feature engineering part
segmentation part
---


## Customer Segment

In [ ]:
folder_path = '/Users/E2074/local-datasets/customer/segmentation/consideration/' + city + '/segment/'

# Get all parquet files in the folder
parquet_files = glob.glob(os.path.join(folder_path, '*.parquet'))

# Read and concatenate all parquet files into a single DataFrame
df_segment_concat = pd.concat([pd.read_parquet(file) for file in parquet_files], ignore_index=True)

In [ ]:
df_segment_concat.head(1)

In [ ]:
df_segment_concat.shape

In [ ]:
df_segment_concat.segment.unique()

## Stability

In [ ]:
# Reducing df memory 
df_segment_concat = df_segment_concat[['run_date', 'customer_id', 'segment', 'updated_segment']]

In [ ]:
df_segment_concat.run_date.unique()

In [ ]:
df_base = df_segment_concat[df_segment_concat['run_date'].isin(['2025-03-03'])].reset_index(drop=True)
df_base

In [ ]:
df_stability_window = df_segment_concat[~df_segment_concat['run_date'].isin(['2025-03-03'])].sort_values(by=['run_date']).reset_index(drop=True)
df_stability_window

In [ ]:
df_stability_window.run_date.unique()

In [ ]:
df_merge = pd.merge(df_base, df_stability_window, how='left', on='customer_id')
df_merge

In [ ]:
df_merge_pivot = df_merge.groupby(['run_date_x', 'run_date_y', 'updated_segment_x', 'updated_segment_y']).agg(customers=('customer_id','nunique')).reset_index()

In [ ]:
df_merge_pivot

In [ ]:
df_merge_pivot.pivot(index = ['run_date_x', 'updated_segment_x', 'updated_segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0)

In [ ]:
df_base.groupby(['run_date', 'updated_segment']).agg(customers=('customer_id','nunique')).reset_index()

## Previous Segment Stability

### taxi_regularity_segment

In [8]:
run_date

['2025-03-03',
 '2025-03-10',
 '2025-03-17',
 '2025-03-24',
 '2025-03-31',
 '2025-04-07',
 '2025-04-14',
 '2025-04-21',
 '2025-04-28',
 '2025-05-05']

In [17]:
def get_previous_segment_customer_base(run_date):
    
    base_query = f"""
    
    select 
        run_date, 
        customer_id, 
        taxi_regularity_segment
        -- taxi_fe_regularity_segment 
    from 
        datasets.iallocator_customer_segments 
    where 
        run_date = date_format(date('{run_date}'), '%Y-%m-%d')
        and taxi_lifetime_last_ride_city = '{city_name}'
        and taxi_regularity_segment != 'UNKNOWN' 
    """
    
    df_base_query = pd.read_sql(base_query, conn3)
    df_base_query['run_date'] = run_date
    print('executed - ', run_date)

    df_base_query.to_parquet(local_path + city + '/prev_segment/prev_customer_base_{}.parquet'.format(run_date), index=False)

In [18]:
for i in run_date[3:]:
    
    print('starting - ', i)
    get_previous_segment_customer_base(i)

starting -  2025-03-24
executed -  2025-03-24
starting -  2025-03-31
executed -  2025-03-31
starting -  2025-04-07
executed -  2025-04-07
starting -  2025-04-14
executed -  2025-04-14
starting -  2025-04-21
executed -  2025-04-21
starting -  2025-04-28
executed -  2025-04-28
starting -  2025-05-05
executed -  2025-05-05


In [19]:
folder_path = '/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/prev_segment'

# Get all parquet files in the folder
prev_parquet_files = glob.glob(os.path.join(folder_path, '*.parquet'))

# Read and concatenate all parquet files into a single DataFrame
df_previous_segment_concat = pd.concat([pd.read_parquet(file) for file in prev_parquet_files], ignore_index=True)

In [20]:
df_previous_segment_concat.head(1)

,run_date,customer_id,taxi_regularity_segment
0,2025-03-31,63621000192f9565bfe3c91a,DAILY


In [21]:
df_previous_segment_concat.shape

(60488342, 3)

### Stability

In [22]:
df_previous_segment_concat.run_date.unique()

array(['2025-03-31', '2025-03-17', '2025-04-14', '2025-04-28',
       '2025-04-07', '2025-04-21', '2025-03-10', '2025-05-05',
       '2025-03-03', '2025-03-24'], dtype=object)

In [23]:
df_base_prev = df_previous_segment_concat[df_previous_segment_concat['run_date'].isin(['2025-03-03'])].reset_index(drop=True)
df_base_prev

,run_date,customer_id,taxi_regularity_segment
0,2025-03-03,65cc9f5d107646988ca7cbfe,MONTHLY
1,2025-03-03,5fec62a5ee2e9cf63c81f3d0,WEEKLY
2,2025-03-03,6260f2fb2c04373d294067d4,RARE_NEED
3,2025-03-03,63d8ae8f6b21d714418dfa55,WEEKLY
4,2025-03-03,659e4952b3bae65867b994db,RARE_NEED
...,...,...,...
5729986,2025-03-03,66ece87d7a4c8a6141bac9a9,RARE_NEED
5729987,2025-03-03,61b2daeebd965e1f5fbcefa3,RARE_NEED
5729988,2025-03-03,6460a8f39bb65e1f9021ee64,MONTHLY
5729989,2025-03-03,61585820c94e7965abcd3146,RARE_NEED


In [24]:
df_stability_window_prev = df_previous_segment_concat[~df_previous_segment_concat['run_date'].isin(['2025-03-03'])].sort_values(by=['run_date']).reset_index(drop=True)
df_stability_window_prev

,run_date,customer_id,taxi_regularity_segment
0,2025-03-10,6720a5f8b0cf4681ad13ec44,BI_WEEKLY
1,2025-03-10,61cf75d1a75746bfa8e44619,RARE_NEED
2,2025-03-10,620b355057d2b2ceeeafedf8,RARE_NEED
3,2025-03-10,64aecfb77fe9533b3ff6af57,MONTHLY
4,2025-03-10,61b867c417dd6a10c413e7fa,WEEKLY
...,...,...,...
54758346,2025-05-05,66abf5a7c9ff56fcce918df4,WEEKLY
54758347,2025-05-05,62c2718c5bce425623ae2591,RARE_NEED
54758348,2025-05-05,5e6b3350f5667d7cb3239387,RARE_NEED
54758349,2025-05-05,64d5e6e412dbf310dfa2a262,RARE_NEED


In [25]:
df_stability_window_prev.run_date.unique()

array(['2025-03-10', '2025-03-17', '2025-03-24', '2025-03-31',
       '2025-04-07', '2025-04-14', '2025-04-21', '2025-04-28',
       '2025-05-05'], dtype=object)

In [26]:
df_merge_prev = pd.merge(df_base_prev, df_stability_window_prev, how='left', on='customer_id')
df_merge_prev

,run_date_x,customer_id,taxi_regularity_segment_x,run_date_y,taxi_regularity_segment_y
0,2025-03-03,65cc9f5d107646988ca7cbfe,MONTHLY,2025-03-10,MONTHLY
1,2025-03-03,65cc9f5d107646988ca7cbfe,MONTHLY,2025-03-17,MONTHLY
2,2025-03-03,65cc9f5d107646988ca7cbfe,MONTHLY,2025-03-24,MONTHLY
3,2025-03-03,65cc9f5d107646988ca7cbfe,MONTHLY,2025-03-31,MONTHLY
4,2025-03-03,65cc9f5d107646988ca7cbfe,MONTHLY,2025-04-07,MONTHLY
...,...,...,...,...,...
50084312,2025-03-03,625d223d00b5f41ac4b124c2,RARE_NEED,2025-04-07,RARE_NEED
50084313,2025-03-03,625d223d00b5f41ac4b124c2,RARE_NEED,2025-04-14,RARE_NEED
50084314,2025-03-03,625d223d00b5f41ac4b124c2,RARE_NEED,2025-04-21,RARE_NEED
50084315,2025-03-03,625d223d00b5f41ac4b124c2,RARE_NEED,2025-04-28,RARE_NEED


In [27]:
df_merge_pivot_prev = df_merge_prev.groupby(['run_date_x', 'run_date_y', 'taxi_regularity_segment_x', 'taxi_regularity_segment_y']).agg(customers=('customer_id','nunique')).reset_index()
df_merge_pivot_prev

,run_date_x,run_date_y,taxi_regularity_segment_x,taxi_regularity_segment_y,customers
0,2025-03-03,2025-03-10,BI_WEEKLY,BI_WEEKLY,756971
1,2025-03-03,2025-03-10,BI_WEEKLY,DAILY,9220
2,2025-03-03,2025-03-10,BI_WEEKLY,MONTHLY,115954
3,2025-03-03,2025-03-10,BI_WEEKLY,WEEKLY,77517
4,2025-03-03,2025-03-10,DAILY,BI_WEEKLY,5320
5,2025-03-03,2025-03-10,DAILY,DAILY,313881
6,2025-03-03,2025-03-10,DAILY,WEEKLY,14562
7,2025-03-03,2025-03-10,MONTHLY,BI_WEEKLY,109612
8,2025-03-03,2025-03-10,MONTHLY,MONTHLY,660047
9,2025-03-03,2025-03-10,MONTHLY,RARE_NEED,104682


In [28]:
df_merge_pivot_prev.pivot(index = ['run_date_x', 'taxi_regularity_segment_x', 'taxi_regularity_segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0)

customers  \
run_date_y                                                     2025-03-10   
run_date_x taxi_regularity_segment_x taxi_regularity_segment_y              
2025-03-03 BI_WEEKLY                 BI_WEEKLY                   756971.0   
                                     DAILY                         9220.0   
                                     MONTHLY                     115954.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                       77517.0   
           DAILY                     BI_WEEKLY                     5320.0   
                                     DAILY                       313881.0   
                                     MONTHLY                          0.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                       14562.0   
           MONTHLY                   BI_WEEKLY                   109612.0   
                                     DAILY                            0.0   
                                     MONTHLY                     660047.0   
                                     RARE_NEED                   104682.0   
                                     WEEKLY                           0.0   
           RARE_NEED                 BI_WEEKLY                        0.0   
                                     DAILY                            0.0   
                                     MONTHLY                      82384.0   
                                     RARE_NEED                  2716145.0   
                                     WEEKLY                           0.0   
           WEEKLY                    BI_WEEKLY                    68762.0   
                                     DAILY                        18447.0   
                                     MONTHLY                          0.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                      614960.0   

                                                                           \
run_date_y                                                     2025-03-17   
run_date_x taxi_regularity_segment_x taxi_regularity_segment_y              
2025-03-03 BI_WEEKLY                 BI_WEEKLY                   633212.0   
                                     DAILY                        16382.0   
                                     MONTHLY                     185302.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                      113711.0   
           DAILY                     BI_WEEKLY                     9866.0   
                                     DAILY                       296106.0   
                                     MONTHLY                          0.0   
                                     RARE_NEED                        0.0   
                                     WEEKLY                       23890.0   
           MONTHLY                   BI_WEEKLY                   153905.0   
                                     DAILY                            0.0   
                                     MONTHLY                     527646.0   
                                     RARE_NEED                   184614.0   
                                     WEEKLY                           0.0   
           RARE_NEED                 BI_WEEKLY                    16567.0   
                                     DAILY                            0.0   
                                     MONTHLY                     130372.0   
                                     RARE_NEED                  2639717.0   
                                     WEEKLY                           0.0   
           WEEKLY                    BI_WEEKLY                    96637.0   
                                     DAILY                 

In [91]:
df_merge_pivot_prev.pivot(index = ['run_date_x', 'taxi_regularity_segment_x', 'taxi_regularity_segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0).to_clipboard(index=False)

In [30]:
df_base_prev.groupby(['run_date', 'taxi_regularity_segment']).agg(customers=('customer_id','nunique')).reset_index()

,run_date,taxi_regularity_segment,customers
0,2025-03-03,BI_WEEKLY,974871
1,2025-03-03,DAILY,340655
2,2025-03-03,MONTHLY,884696
3,2025-03-03,RARE_NEED,2812984
4,2025-03-03,WEEKLY,716785


### FE Regularity

In [31]:
run_date

['2025-03-03',
 '2025-03-10',
 '2025-03-17',
 '2025-03-24',
 '2025-03-31',
 '2025-04-07',
 '2025-04-14',
 '2025-04-21',
 '2025-04-28',
 '2025-05-05']

In [37]:
def get_previous_segment_customer_base(run_date):
    
    base_query = f"""
    
    select 
        run_date, 
        customer_id, 
        -- taxi_regularity_segment
        taxi_fe_regularity_segment 
    from 
        datasets.iallocator_customer_segments 
    where 
        run_date = date_format(date('{run_date}'), '%Y-%m-%d')
        and taxi_lifetime_last_ride_city = '{city_name}'
        and taxi_fe_regularity_segment != 'UNKNOWN' 
    """
    
    df_base_query = pd.read_sql(base_query, conn4)
    df_base_query['run_date'] = run_date
    print('executed - ', run_date)

    df_base_query.to_parquet(local_path + city + '/prev_segment2/prev_customer_base_{}.parquet'.format(run_date), index=False)

In [38]:
for i in run_date[3:]:
    
    print('starting - ', i)
    get_previous_segment_customer_base(i)

starting -  2025-03-24
executed -  2025-03-24
starting -  2025-03-31
executed -  2025-03-31
starting -  2025-04-07
executed -  2025-04-07
starting -  2025-04-14
executed -  2025-04-14
starting -  2025-04-21
executed -  2025-04-21
starting -  2025-04-28
executed -  2025-04-28
starting -  2025-05-05
executed -  2025-05-05


In [39]:
folder_path = '/Users/E2074/local-datasets/customer/segmentation/consideration/delhi/prev_segment2'

# Get all parquet files in the folder
prev_parquet_files = glob.glob(os.path.join(folder_path, '*.parquet'))

# Read and concatenate all parquet files into a single DataFrame
df_previous_segment_concat = pd.concat([pd.read_parquet(file) for file in prev_parquet_files], ignore_index=True)

In [40]:
df_previous_segment_concat.head(1)

,run_date,customer_id,taxi_fe_regularity_segment
0,2025-03-31,66df535ad8b3ee466a1c6c4e,MONTHLY


In [41]:
df_previous_segment_concat.shape

(45518135, 3)

### Stability

In [42]:
df_previous_segment_concat.run_date.unique()

array(['2025-03-31', '2025-03-17', '2025-04-14', '2025-04-28',
       '2025-04-07', '2025-04-21', '2025-03-10', '2025-05-05',
       '2025-03-03', '2025-03-24'], dtype=object)

In [43]:
df_base_prev = df_previous_segment_concat[df_previous_segment_concat['run_date'].isin(['2025-03-03'])].reset_index(drop=True)
df_base_prev

,run_date,customer_id,taxi_fe_regularity_segment
0,2025-03-03,6612d04a4f976baed5b6ed89,MONTHLY
1,2025-03-03,65d40897c48581191d416f97,RARE_NEED
2,2025-03-03,63e9085f2217a7dbecd70b3b,DAILY
3,2025-03-03,652bb8803246660a488d28fc,BI_WEEKLY
4,2025-03-03,657b412344fc61de29b74089,RARE_NEED
...,...,...,...
4325313,2025-03-03,6266621e1d2d6f55e5582342,MONTHLY
4325314,2025-03-03,646850739dbf7d77a8841279,RARE_NEED
4325315,2025-03-03,63ea2413ae330d6058148d4e,BI_WEEKLY
4325316,2025-03-03,63c25a1a1438dfac9d0f80e9,WEEKLY


In [44]:
df_stability_window_prev = df_previous_segment_concat[~df_previous_segment_concat['run_date'].isin(['2025-03-03'])].sort_values(by=['run_date']).reset_index(drop=True)
df_stability_window_prev

,run_date,customer_id,taxi_fe_regularity_segment
0,2025-03-10,661d53e646c9a416e9faab21,BI_WEEKLY
1,2025-03-10,6695d262d426be574760308b,BI_WEEKLY
2,2025-03-10,650977c81213a91b0b84f758,WEEKLY
3,2025-03-10,5fd0821b0141471976127cbb,BI_WEEKLY
4,2025-03-10,66445bb0510947f10864bfaf,BI_WEEKLY
...,...,...,...
41192812,2025-05-05,67cd0756cff2e363a50aff22,BI_WEEKLY
41192813,2025-05-05,64d4a896da3d2b05c1825b3d,MONTHLY
41192814,2025-05-05,665ca8d399dbb5fe9719b4da,RARE_NEED
41192815,2025-05-05,67a09514197b899daab6fa8b,MONTHLY


In [45]:
df_stability_window_prev.run_date.unique()

array(['2025-03-10', '2025-03-17', '2025-03-24', '2025-03-31',
       '2025-04-07', '2025-04-14', '2025-04-21', '2025-04-28',
       '2025-05-05'], dtype=object)

In [46]:
df_merge_prev = pd.merge(df_base_prev, df_stability_window_prev, how='left', on='customer_id')
df_merge_prev

,run_date_x,customer_id,taxi_fe_regularity_segment_x,run_date_y,taxi_fe_regularity_segment_y
0,2025-03-03,6612d04a4f976baed5b6ed89,MONTHLY,2025-03-10,MONTHLY
1,2025-03-03,6612d04a4f976baed5b6ed89,MONTHLY,2025-03-17,MONTHLY
2,2025-03-03,6612d04a4f976baed5b6ed89,MONTHLY,2025-03-24,MONTHLY
3,2025-03-03,6612d04a4f976baed5b6ed89,MONTHLY,2025-03-31,MONTHLY
4,2025-03-03,6612d04a4f976baed5b6ed89,MONTHLY,2025-04-07,MONTHLY
...,...,...,...,...,...
32822619,2025-03-03,63c25a1a1438dfac9d0f80e9,WEEKLY,2025-05-05,BI_WEEKLY
32822620,2025-03-03,663df36f802b4ef8fd320bac,RARE_NEED,2025-03-10,RARE_NEED
32822621,2025-03-03,663df36f802b4ef8fd320bac,RARE_NEED,2025-03-17,RARE_NEED
32822622,2025-03-03,663df36f802b4ef8fd320bac,RARE_NEED,2025-03-24,RARE_NEED


In [47]:
df_merge_pivot_prev = df_merge_prev.groupby(['run_date_x', 'run_date_y', 'taxi_fe_regularity_segment_x', 'taxi_fe_regularity_segment_y']).agg(customers=('customer_id','nunique')).reset_index()
df_merge_pivot_prev

,run_date_x,run_date_y,taxi_fe_regularity_segment_x,taxi_fe_regularity_segment_y,customers
0,2025-03-03,2025-03-10,BI_WEEKLY,BI_WEEKLY,1025982
1,2025-03-03,2025-03-10,BI_WEEKLY,DAILY,14281
2,2025-03-03,2025-03-10,BI_WEEKLY,MONTHLY,140612
3,2025-03-03,2025-03-10,BI_WEEKLY,WEEKLY,68539
4,2025-03-03,2025-03-10,DAILY,BI_WEEKLY,9005
5,2025-03-03,2025-03-10,DAILY,DAILY,377031
6,2025-03-03,2025-03-10,DAILY,WEEKLY,16199
7,2025-03-03,2025-03-10,MONTHLY,BI_WEEKLY,157631
8,2025-03-03,2025-03-10,MONTHLY,MONTHLY,1009967
9,2025-03-03,2025-03-10,MONTHLY,RARE_NEED,128907


In [48]:
df_merge_pivot_prev.pivot(index = ['run_date_x', 'taxi_fe_regularity_segment_x', 'taxi_fe_regularity_segment_y'] , columns = ['run_date_y'], values =['customers']).fillna(0)

customers  \
run_date_y                                                           2025-03-10   
run_date_x taxi_fe_regularity_segment_x taxi_fe_regularity_segment_y              
2025-03-03 BI_WEEKLY                    BI_WEEKLY                     1025982.0   
                                        DAILY                           14281.0   
                                        MONTHLY                        140612.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                          68539.0   
           DAILY                        BI_WEEKLY                        9005.0   
                                        DAILY                          377031.0   
                                        MONTHLY                             0.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                          16199.0   
           MONTHLY                      BI_WEEKLY                      157631.0   
                                        DAILY                               0.0   
                                        MONTHLY                       1009967.0   
                                        RARE_NEED                      128907.0   
                                        WEEKLY                              0.0   
           RARE_NEED                    BI_WEEKLY                           0.0   
                                        DAILY                               0.0   
                                        MONTHLY                         93600.0   
                                        RARE_NEED                      572515.0   
                                        WEEKLY                              0.0   
           WEEKLY                       BI_WEEKLY                       61746.0   
                                        DAILY                           18657.0   
                                        MONTHLY                             0.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                         428454.0   

                                                                                 \
run_date_y                                                           2025-03-17   
run_date_x taxi_fe_regularity_segment_x taxi_fe_regularity_segment_y              
2025-03-03 BI_WEEKLY                    BI_WEEKLY                      895585.0   
                                        DAILY                           24925.0   
                                        MONTHLY                        212885.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                         102870.0   
           DAILY                        BI_WEEKLY                       16101.0   
                                        DAILY                          355258.0   
                                        MONTHLY                             0.0   
                                        RARE_NEED                           0.0   
                                        WEEKLY                          25998.0   
           MONTHLY                      BI_WEEKLY                      242861.0   
                                        DAILY                               0.0   
                                        MONTHLY                        797401.0   
                                        RARE_NEED                      199202.0   
                                        WEEKLY                              0.0   
           RARE_NEED                    BI_WEEKLY                         734.0   
                                        DAILY                               0.0   
                                        MONTHLY                        136571.0   
  

In [49]:
df_base_prev.groupby(['run_date', 'taxi_fe_regularity_segment']).agg(customers=('customer_id','nunique')).reset_index()

,run_date,taxi_fe_regularity_segment,customers
0,2025-03-03,BI_WEEKLY,1267511
1,2025-03-03,DAILY,410518
2,2025-03-03,MONTHLY,1345532
3,2025-03-03,RARE_NEED,783225
4,2025-03-03,WEEKLY,518532
